# Verify complete spatial source rows, exact-ID mappings and microscopy provenance


Replay complete marker tables against original files and installed IDs. Preserve diagnostic v1; v2 fixes an attachment-discovery failure without changing any marker rows. Verify all media MD5/SHA receipts and candidate-only microscopy semantics. This is audit verification, not a biological accuracy evaluation.

In [1]:
import json, hashlib
import pandas as pd
from pathlib import Path
from starplast import organisms as O
base=Path('results/spatial_truth_source_review_2026_10_08')
reports=json.loads((base/'table_audit_v2/table_review.json').read_text())
sources={r['source_id']:r for r in json.loads((base/'sources.json').read_text())}
for report in reports:
    key=report['source_id']
    source=sources[key]
    organism=O.TOXOPLASMA if key=='lopit_tgon' else O.FALCIPARUM
    reference=set(pd.read_parquet(O.nodes_path(organism),columns=['gene_id']).gene_id.astype(str))
    binding=source['existing_file'] if key=='pf_spatial_proteome' else source['companions'][0]['file']
    sheet='PlasmoLOPIT_data_summary' if key=='pf_spatial_proteome' else 'S3 - Feature metadata'
    idcol='Gene accession' if key=='pf_spatial_proteome' else 'Accession'
    raw=pd.read_excel(binding['path'],sheet_name=sheet,dtype=str)
    assert hashlib.sha256(Path(binding['path']).read_bytes()).hexdigest()==binding['sha256']
    for field in report['fields']:
        name=key+'_'+field['context']+'_rows.parquet'
        rows=pd.read_parquet(base/'table_audit_v2'/name)
        pd.testing.assert_frame_equal(rows,pd.read_parquet(base/'table_audit'/name))
        assert rows.accession.tolist()==raw[idcol].tolist()
        assert rows.source_row.tolist()==list(range(2,len(raw)+2))
        assert rows.marker_label.fillna('').tolist()==raw[field['marker_field']].fillna('').tolist()
        assert rows.source_prediction.fillna('').tolist()==raw[rows.prediction_field.iloc[0]].fillna('').tolist()
        assert rows.exact_installed_id.tolist()==raw[idcol].isin(reference).tolist()
        assert int(rows.marker_present.sum())==field['marker_rows']
        assert rows[rows.marker_present].marker_label.value_counts().to_dict()==field['marker_class_counts']
        assert not rows.benchmark_admitted.any() and set(rows.truth_grade)=={'unresolved'}
        print(key,field['context'],'all rows retained',len(rows),'markers',field['marker_rows'],'unmapped marker IDs',rows.loc[rows.marker_present & ~rows.exact_installed_id,'accession'].tolist())

lopit_tgon extracellular_tachyzoites all rows retained 3832 markers 718 unmapped marker IDs ['TGME49_253900', 'TGME49_304955']
pf_spatial_proteome late_schizonts_S1_S2 all rows retained 3000 markers 542 unmapped marker IDs []
pf_spatial_proteome schizonts_plus_merozoites_S1_S2_S3 all rows retained 3000 markers 484 unmapped marker IDs []


Initial verification required the literal unknown sentinel in both marker fields and correctly stopped on one missing S1/S2/S3 field. Preserve the original verifier in code/verify_spatial_review_initial.py. Missing and unknown marker values both exclude a gene from the training marker set; the revised check accepts those two distinct absence representations without changing any saved data.

In [2]:
media=json.loads((base/'table_audit_v2/media_receipts.json').read_text())
assert len(media)==4
for row in media:
    assert row['status']=='retrieved_verified_primary_media'
    data=Path(row['file']['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest()==row['file']['sha256']
    assert hashlib.md5(data).hexdigest()==row['published_md5']
outcomes=json.loads((base/'microscopy_review/outcomes.json').read_text())
assert len(outcomes)==15 and len({r['gene_id'] for r in outcomes})==15
positive=[r for r in outcomes if r['IFA_detected'] is True]
unknown=[r for r in outcomes if r['IFA_detected'] is None]
assert len(positive)==9 and len(unknown)==6
assert all(r['exact_installed_id'] and r['protein_in_spatial_table'] for r in positive)
assert all(r['experimental_compartment'] is None for r in unknown)
assert all(r['markers (S1-S2)'] in (None,'unknown') and r['markers (S1-S2-S3)'] in (None,'unknown') for r in positive)
assert all(not r['benchmark_admitted'] for r in outcomes)
print('Four primary media hashes exact; nine observed IFA genes outside both marker fields; six unknown outcomes preserved; zero biological admissions.')

Four primary media hashes exact; nine observed IFA genes outside both marker fields; six unknown outcomes preserved; zero biological admissions.


Interpretation: Toxoplasma 718 final markers include the same paper's 62 new IFA results, so those results do not independently validate the published final classifier. Plasmodium Final location (S1-S2) incorporates rhoptry assignments from the three-way S1/S2/S3 analysis; its field name must not imply every final assignment comes only from schizonts. Literal marker/prediction disagreements reflect unresolved class granularity and are not scored as biological mistakes. Nine selected Plasmodium IFA outcomes are a promising separate assay candidate, requiring reviewed taxonomy/source exclusions/context before admission.